# A line through a posterior is a claim of certainty

Somebody will screenshot the response curve. It will end up on a slide, next to a proposed
budget, without any of the text around it — and a curve drawn as a line says *this is the
response*, which is a claim the model never made. Whether the next dose is worth anything is
often a question the fit genuinely cannot answer, and a line has no way to say so.

A response curve is a **posterior quantity**. `forward(surface, dose, theta)` evaluated
at the posterior mean of the parameters is a line, and a line says the curve is known
when it is not — for a nonlinear surface it is not even the curve the model believes,
because the mean of a nonlinear function is not the function of the mean.

`ResponseBand` is that curve as data: a dose grid, the posterior mean and median at
every grid point, and the interval at every grid point **with the definition and mass
that produced it**. `response_band` builds one for the expected outcome and
`marginal_band` for the derivative. `viz.response_curve` and `viz.marginal_curve`
render exactly that object, so there is one implementation of "evaluate the surface
through the draws" and no way to draw a surface figure without its band.

In [ ]:
import numpy as np

from axiom.core import Interval, TimeWindow, is_failure
from axiom.sim import DosePlan, surface_world
from axiom.surface import (
    BandKind, HillKernel, ResponseBand, SupportsBands, fit, forward, marginal_band, response_band,
)
from axiom.viz import marginal_curve, response_curve

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, curve_band, mark_y, shade

enable();  # every axiom result renders itself from here on

world = surface_world(
    n_units=6, n_periods=16, treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    doses=DosePlan(scale=50.0, spread=1.0, zero_fraction=0.2),
    intercept="shared", noise_sd=0.6, seed=4,
)
result = fit(world.spec, world.panel, backend="laplace", draws=800, chains=1, seed=4)
print("converged:", result.converged, "| draws:", result.n_draws)
print("a fit satisfies the band protocol:", isinstance(result, SupportsBands))

## The band, and what it is a band of

At each grid dose the intervention sets that treatment to the level everywhere in the
window and leaves the others at their observed doses; each posterior draw goes through
the same `forward` the likelihood used, and the draw's value is the mean over units and
periods. So this is uncertainty in the **expected** outcome at that dose, not a
predictive interval for one unit — `kind` records which.

In [ ]:
band = response_band(result, "a", n_grid=9, mass=0.9, definition="eti")
assert isinstance(band, ResponseBand)
kind: BandKind = band.kind
print(f"kind={kind}  draws={band.n_draws}  {band.label()}  dimension={band.dimension}")
print(f"{'dose':>8} {'mean':>8} {'median':>8} {'lower':>8} {'upper':>8} {'width':>7}")
table(
    [
        [f"{band.doses[i]:.1f}", f"{band.mean[i]:.3f}", f"{band.median[i]:.3f}",
         f"{band.lower[i]:.3f}", f"{band.upper[i]:.3f}", f"{band.width[i]:.3f}"]
        for i in range(band.n_grid)
    ],
    headers=("dose", "mean", "median", "lower", "upper", "width"),
)
interval: Interval = band.interval_at(band.n_grid - 1)
print("\nthe interval at the top dose, carrying its own provenance:", interval)
print("axis titles for whatever draws it:", band.axis_titles())

### Why not the curve at the posterior mean

Evaluating `forward` at the mean of the parameters is a different quantity from the
mean of `forward` over the posterior — the mean of a nonlinear function is not the
function of the mean — and the two curves do differ. But on a fit this well determined
the gap is small next to the band's own width, and *that* is the more useful
comparison: the point curve is not badly located, it is missing the width entirely.

In [ ]:
theta_mean = {p.name: float(result.posterior.summary(p.name).mean)
              for p in result.surface.model.parameters if p.name != "sigma"}
grid = np.asarray(band.doses)
at_mean = np.ravel(forward(result.surface, {"a": np.tile(grid, (result.n_units, 1))}, theta_mean)).reshape(
    result.n_units, -1).mean(axis=0)
print(f"{'dose':>8} {'E[f(theta)]':>12} {'f(E[theta])':>12} {'gap':>8} {'band width':>11}")
table(
    [
        [f"{dose:.1f}", f"{band.mean[i]:.3f}", f"{at_mean[i]:.3f}",
         f"{band.mean[i] - at_mean[i]:.3f}", f"{band.width[i]:.3f}"]
        for i, dose in enumerate(grid)
    ],
    headers=("dose", "mean of the curve", "curve at the mean", "gap", "band width"),
)

In [ ]:
fig = curve_band(
    grid, np.asarray(band.mean), np.asarray(band.lower), np.asarray(band.upper),
    label="the posterior",
    title="The same fit, drawn two ways",
    subtitle=f"{band.label()} around the mean of the curve, against the curve at the mean of the parameters",
    x_title="dose of a", y_title="expected outcome",
)
curve_band(grid, at_mean, label="curve at the posterior mean", color=ORANGE, dash="dot", fig=fig)
caption(fig, "The two lines are close. That is not the point — the point is that one of them "
             "comes with the shaded region and the other does not, and only the shaded region "
             "answers 'could the response at the top dose be half of what you just told me?'")

## The marginal band is the one a dose decision reads

`d outcome / d dose` — whether the next unit of dose is worth anything. Its **sign** is
the thing worth an interval: where the band straddles zero the model does not know
whether to titrate up or down, and a line through the posterior mean will not say so.

In [ ]:
slope = marginal_band(result, "a", n_grid=9, mass=0.9)
assert not is_failure(slope)
print(f"kind={slope.kind}  dimension={slope.dimension}  y-axis: {slope.axis_titles()[1]}")
print(f"{'dose':>8} {'slope':>9} {'lower':>9} {'upper':>9}  straddles zero?")
table(
    [
        [f"{slope.doses[i]:.1f}", f"{slope.mean[i]:.4f}", f"{slope.lower[i]:.4f}",
         f"{slope.upper[i]:.4f}", "yes" if slope.lower[i] <= 0.0 <= slope.upper[i] else "no"]
        for i in range(slope.n_grid)
    ],
    headers=("dose", "slope", "lower", "upper", "straddles zero"),
)

In [ ]:
straddling = [i for i in range(slope.n_grid) if slope.lower[i] <= 0.0 <= slope.upper[i]]
fig = curve_band(
    np.asarray(slope.doses), np.asarray(slope.mean), np.asarray(slope.lower), np.asarray(slope.upper),
    label="d outcome / d dose",
    title="Where the model does not know whether to spend more",
    subtitle=f"{slope.label()} on the marginal effect — the sign is the decision",
    x_title="dose of a", y_title="marginal effect per unit dose",
)
mark_y(fig, 0.0, text="no effect")
if straddling:
    shade(fig, float(slope.doses[straddling[0]]), float(slope.doses[straddling[-1]]),
          text="band contains zero", color=ORANGE, alpha=0.10)
caption(fig, "Inside the shaded doses the posterior admits that the next unit of dose might "
             "do nothing, or might do harm. A point estimate of the slope is positive "
             "everywhere in that region and would have been quoted as such.")

## The figures

Both take either a fit or a band already built, so a report that computed the band once
does not compute it again. There is no argument on either that turns the band off.

In [ ]:
figure = response_curve(result, "a", n_grid=25, mass=0.9)
print("response_curve traces:", len(figure.data),
      "| first is the band:", figure.data[0].fill == "toself", "|", figure.data[0].name)
print("marginal_curve traces:", len(marginal_curve(result, "a", n_grid=25).data))
from_band = response_curve(band)
print("drawn from a prebuilt band:", len(from_band.data), "traces")
window = TimeWindow(start=0, stop=8)
print("restricted to a window:", response_band(result, "a", n_grid=4, window=window).n_grid, "points")
figure

## What this notebook decided

- A curve is a posterior quantity, so it is returned as a `ResponseBand` — grid, mean,
  median and an `Interval` at every point — not as an array of numbers.
- `E[f(theta)]` and `f(E[theta])` are different curves, and here the gap between them
  is an order of magnitude smaller than the band width. The problem with a point curve
  is not that it is in the wrong place — it is that it claims a width of zero.
- The marginal band is where a dose decision lives, and its interval straddling zero is
  the answer "the model does not know", which a point estimate cannot express.
- `viz` renders bands and nothing else, so a surface figure without uncertainty is not
  a thing this library can produce. `tests/contracts/test_surface_uncertainty.py` is
  the gate.

That last point is the one worth taking seriously. There is no keyword anywhere in this
package that draws a response curve without its band, because the screenshot at the top of
this notebook is going to happen and the only defence is that the honest figure is the only
figure available.